# Notebook de Normalização
Este notebook aplica a normalização da base PEDE, padroniza nomes de colunas, cria features derivadas (IPP, pedra_22_num, evolucao_pedra, sent_score, fase) e salva os artefatos: CSV limpo em `data/pe_de_clean.csv` e `scaler.pkl` em `app/`.

Observação: ajuste `DATA_PATH` se você colocará o CSV em outra pasta local.

In [ ]:
# Imports e configurações
import re
import unicodedata
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib

# Caminhos
ROOT = Path('/workspaces/passos_magicos')
# O CSV original pode estar na raiz e também em data/; ajustamos automaticamente
CANDIDATE = [ROOT / 'data' / 'BASE DE DADOS PEDE 2024 - DATATHON - PEDE2022.csv', ROOT / 'BASE DE DADOS PEDE 2024 - DATATHON - PEDE2022.csv']
DATA_PATH = None
for p in CANDIDATE:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError('CSV da base PEDE não encontrado nos caminhos esperados. Coloque o arquivo em /data ou na raiz.')

OUT_CSV = ROOT / 'data' / 'pe_de_clean.csv'
SCALER_PATH = ROOT / 'app' / 'scaler.pkl'
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.exists(), SCALER_PATH.parent.mkdir(parents=True, exist_ok=True)

print('DATA_PATH =', DATA_PATH)

In [ ]:
# Função utilitária para slugificar nomes de colunas
def slugify_column(name: str) -> str:
    # remove acentos, minúsculas, substitui espaços e caracteres não alfanuméricos por underscore
    if not isinstance(name, str):
        return name
    s = unicodedata.normalize('NFKD', name)
    s = ''.join([c for c in s if not unicodedata.combining(c)])
    s = s.lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

# Leitura do CSV e normalização de colunas
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
orig_cols = df.columns.tolist()
df.columns = [slugify_column(c) for c in df.columns]
print('Colunas originais (exemplos):', orig_cols[:10])
print('Colunas normalizadas (exemplos):', df.columns.tolist()[:10])

In [ ]:
# Mapear colunas específicas e criar features derivadas
# Estas operações assumem nomes de colunas padronizados para o dataset de exemplo.
# Ajuste os nomes abaixo se o seu CSV tiver colunas com nomes diferentes após slugify.

# Exemplo: localizar colunas que contenham 'pedra' seguido por dígitos (ex: pedra_22)
pedra_cols = [c for c in df.columns if c.startswith('pedra')]
print('Encontrado(s) coluna(s) pedra:', pedra_cols)

# Se existir 'pedra_22' ou similar, preferir esse para derivar pedra_22_num
pedra_22_col = None
for c in pedra_cols:
    if '22' in c:
        pedra_22_col = c
        break
if pedra_22_col is None and len(pedra_cols) > 0:
    pedra_22_col = pedra_cols[-1]  # fallback para a última pedra encontrada

# Mapeamento ordinal das pedras (ajuste se houver mais tipos)
pedra_map = {'quartzo': 1, 'agata': 2, 'ametista': 3, 'topazio': 4}  # sem acentos após slugify

if pedra_22_col is not None and pedra_22_col in df.columns:
    # normaliza os valores para lowercase e remove espaços
    df['pedra_22_raw'] = df[pedra_22_col].astype(str).str.lower().str.strip()
    # remover acentos para mapear corretamente
    df['pedra_22_raw'] = df['pedra_22_raw'].str.normalize('NFKD').str.replace(r'[̧̀́̂̃]', '', regex=True)
    df['pedra_22_num'] = df['pedra_22_raw'].map(pedra_map).fillna(0).astype(int)
else:
    df['pedra_22_num'] = 0

# Exemplo de criação de IPP (ajuste conforme colunas específicas: cf, ct)
possible_cf = [c for c in df.columns if 'cf' in c]  # heurística
possible_ct = [c for c in df.columns if 'ct' in c]
if possible_cf and possible_ct:
    cf_col = possible_cf[0]
    ct_col = possible_ct[0]
    df['ipp'] = df[[cf_col, ct_col]].mean(axis=1)
else:
    # fallback: criar ipp a partir de colunas conhecidas; aqui colocamos NaN para revisão manual
    df['ipp'] = np.nan

# Evolução de pedra: se houver pedra_21 e pedra_22 (exemplos), criar diferença ordinal
pedra_21_col = None
for c in pedra_cols:
    if '21' in c:
        pedra_21_col = c
        break
if pedra_21_col and pedra_22_col and pedra_21_col in df.columns and pedra_22_col in df.columns:
    df['pedra_21_raw'] = df[pedra_21_col].astype(str).str.lower().str.strip()
    df['pedra_21_raw'] = df['pedra_21_raw'].str.normalize('NFKD').str.replace(r'[̧̀́̂̃]', '', regex=True)
    df['pedra_21_num'] = df['pedra_21_raw'].map(pedra_map).fillna(0).astype(int)
    df['evolucao_pedra'] = df['pedra_22_num'] - df['pedra_21_num']
else:
    df['evolucao_pedra'] = 0

# Campo sent_score (placeholder): se existir coluna de texto com recomendações, usar heurística simples (comprimento do texto / presença de palavras negativas/positivas)
text_cols = [c for c in df.columns if 'destaque' in c or 'recomend' in c or 'obs' in c or 'observ' in c]
if text_cols:
    text_col = text_cols[0]
    df['sent_len'] = df[text_col].astype(str).str.len()
    # heurística simples: score = log1p(len) — placeholder para pipeline de NLP posterior
    df['sent_score'] = np.log1p(df['sent_len'])
else:
    df['sent_score'] = 0.0

# Definir 'fase' se houver coluna que contenha 'fase'
fase_cols = [c for c in df.columns if 'fase' in c]
if fase_cols:
    df['fase'] = df[fase_cols[0]]
else:
    df['fase'] = np.nan

print('Features criadas: pedra_22_num, ipp, evolucao_pedra, sent_score, fase')

In [ ]:
# Selecionar colunas numéricas para escalonamento e salvar scaler
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# remover colunas de id/indice se existirem (heurística)
for bad in ['id', 'aluno', 'matricula']:
    if bad in num_cols:
        try:
            num_cols.remove(bad)
        except ValueError:
            pass

print('Colunas numéricas a escalar (exemplos):', num_cols[:20])
scaler = StandardScaler()
# Substituir NaNs temporariamente pelo valor médio para ajustar o scaler (melhor: imputador dedicado)
df_num = df[num_cols].copy()
df_num_filled = df_num.fillna(df_num.mean())
scaler.fit(df_num_filled)
# aplicar transformação e sobrescrever no dataframe
df[num_cols] = scaler.transform(df_num_filled)

# Salvar artefatos
df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
joblib.dump(scaler, SCALER_PATH)

print('CSV limpo salvo em:', OUT_CSV)
print('Scaler salvo em:', SCALER_PATH)

# Observações e próximos passos
- Este notebook realiza uma normalização inicial e cria colunas derivadas básicas.
- Recomenda-se revisar manualmente os mapeamentos das colunas detectadas (especialmente `cf`/`ct` para IPP e colunas de `pedra_*`).
- Para produção, substitua a heurística de `sent_score` por pipeline NLP (tokenização, embeddings e classifier/regressor de sentimento).
- Após validação, referenciar este notebook no `notebooks/02_modelo_preditivo.ipynb` e executar o smoke-test (pré-processamento + split + salvar scaler).